# 🧠 Training the Tiny Recursive MoE Contrastive (TRMC) Model

Welcome to the interactive training guide for the **TRMC model**. This notebook will walk you through the entire process from environment setup to exporting your model for use in **Ollama**.

### What is TRMC?
TRMC is a high-efficiency architecture (7M-20M parameters) designed for reasoning-heavy tasks. It features:
- **Recursive Core**: Reuses parameters across multiple 'thinking' steps.
- **Sparse MoE**: Provides high capacity with low computational cost.
- **Contrastive Learning**: Aligns latent states for better logic discrimination.
- **Matryoshka Embeddings**: Multi-resolution representations.
- **Vision Awareness**: Ability to process spatial/OCR data.

## 🛠️ 1. Setup & Imports

First, we ensure our environment is correctly configured and import the necessary libraries.

In [ ]:
import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import numpy as np

# Add the parent directory to sys.path to import our custom modules
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

# Also add research folder for local imports
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

from trmc_model import TRMCModel, contrastive_loss
from dataset_curator import TRMCDatasetCurator

print(f"Using device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 🧩 2. Data Preparation

We use a `LogicPuzzlesDataset` that generates sequences and their reversed counterparts. Crucially, it also generates **negative examples** (corrupted sequences) to facilitate contrastive learning.

In [ ]:
class LogicPuzzlesDataset(Dataset):
    """Generates sequence reversal puzzles with contrastive pairs."""
    def __init__(self, size=1000, seq_len=16, vocab_size=32, num_negatives=5):
        self.size = size
        self.seq_len = seq_len
        self.vocab_size = vocab_size
        self.num_negatives = num_negatives
        self.data = []
        
        for _ in range(size):
            # Input sequence (tokens 3 to vocab_size are usable data)
            x = torch.randint(3, vocab_size, (seq_len,))
            # Positive label (reversed sequence)
            y_pos = torch.flip(x, dims=[0])
            
            # Negative labels (corrupted versions of the positive label)
            y_negs = []
            for _ in range(num_negatives):
                y_neg = y_pos.clone()
                # Randomly corrupt ~25% of the sequence
                idx = torch.randint(0, seq_len, (max(1, seq_len // 4),))
                y_neg[idx] = torch.randint(3, vocab_size, (len(idx),))
                y_negs.append(y_neg)
                
            self.data.append((x, y_pos, torch.stack(y_negs)))
            
    def __len__(self):
        return self.size
    
    def __getitem__(self, idx):
        return self.data[idx]

# Initialize Dataset
dataset = LogicPuzzlesDataset(size=2000, seq_len=16, vocab_size=32)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

## 🏗️ 3. Model Initialization

We configure our `TRMCModel`. Note the `matryoshka_dims` which allow the model to learn representations at different granularities.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vocab_size = 32
seq_len = 16
hidden_dim = 128

model = TRMCModel(
    vocab_size=vocab_size, 
    hidden_dim=hidden_dim, 
    num_heads=4, 
    num_experts=8, 
    num_iterations=8,
    matryoshka_dims=[32, 64, 128]
).to(device)

print(f"Model initialized with {sum(p.numel() for p in model.parameters())/1e6:.2f}M parameters.")

## 🚀 4. Training Loop

We use a combined loss: **Cross-Entropy** for next-token prediction and **InfoNCE Contrastive Loss** to ensure the model's reasoning state is robust.

In [ ]:
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
criterion = nn.CrossEntropyLoss()
contrastive_weight = 0.2
epochs = 5

history = {'loss': [], 'ce_loss': [], 'c_loss': []}

model.train()
for epoch in range(epochs):
    epoch_losses = []
    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}")
    
    for x, y_pos, y_negs in progress_bar:
        x, y_pos, y_negs = x.to(device), y_pos.to(device), y_negs.to(device)
        # Dummy vision data for multi-modal path
        images = torch.randn(x.shape[0], 3, 32, 32).to(device)
        
        optimizer.zero_grad()
        
        # Forward pass for the query
        logits, query_latent = model(x, images=images)
        
        # Get latents for positive/negative samples (without gradient for efficiency)
        with torch.no_grad():
            _, pos_latent = model(y_pos, images=images)
            batch_size, num_negs, s_len = y_negs.shape
            _, neg_latent_all = model(y_negs.view(-1, s_len), images=images.repeat_interleave(num_negs, dim=0))
            neg_latents = neg_latent_all.view(batch_size, num_negs, -1, hidden_dim)
            
        # Calculate Losses
        ce_loss = criterion(logits.view(-1, vocab_size), y_pos.view(-1))
        c_loss = contrastive_loss(
            query_latent.mean(1), 
            pos_latent.mean(1), 
            neg_latents.mean(2),
            matryoshka_dims=model.matryoshka_dims
        )
        
        loss = ce_loss + contrastive_weight * c_loss
        
        loss.backward()
        optimizer.step()
        
        # Logging
        history['loss'].append(loss.item())
        history['ce_loss'].append(ce_loss.item())
        history['c_loss'].append(c_loss.item())
        progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})

print("Training Complete!")

## 📈 5. Visualizing Progress

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history['ce_loss'], label='Cross Entropy Loss')
plt.plot(history['c_loss'], label='Contrastive Loss')
plt.title('Training Losses')
plt.xlabel('Steps')
plt.ylabel('Loss')
plt.legend()
plt.show()

## 💾 6. Export for Ollama

To use this model in **Ollama**, we need to:
1. Save the model state dict.
2. Run the conversion script to create a `GGUF` file (handled by our external tool).
3. Create a `Modelfile`.

In [ ]:
# Save the final model
checkpoint_path = "../checkpoints/trmc/model_final.pt"
os.makedirs(os.path.dirname(checkpoint_path), exist_ok=True)
torch.save(model.state_dict(), checkpoint_path)
print(f"Model saved to {checkpoint_path}")

print("\nNext Step: Run the Ollama export tool:")
print("python tools/export_ollama.py --checkpoint checkpoints/trmc/model_final.pt")